# The Naive Definition of Probability

*Following Blitzstein, J. K., & Hwang, J. (2019). Introduction to Probability (2nd ed.), Section 1.3.*

## Index
1. [The Wall Experiment](#1)
2. [Diagram: The Wall, Live](#2)
3. [Example: Counting Outcomes Instead of Area](#3)
4. [Plain English](#4)
5. [Technical Definitions](#5)
6. [References](#6)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display, clear_output

<a id="1"></a>
## 1. The Wall Experiment

Imagine throwing a stone at a wall while blindfolded. The wall is 10 meters wide and 10 meters high, for a total surface of $100\text{ m}^2$.

The **sample space** $S$ is the entire wall: every position where the stone could land. An **event** $A$ is one colored region painted on the wall. If the stone is equally likely to land anywhere, the chance of hitting a region is the ratio of its area to the whole surface:

$$P_{\text{naive}}(A) = \frac{\text{Area}(A)}{\text{Area}(S)}$$

Suppose the wall is painted in three colors:

| Color | Share of the wall | Area |
|---|---|---|
| Red | 50% | $50\text{ m}^2$ |
| Blue | 30% | $30\text{ m}^2$ |
| Green | 20% | $20\text{ m}^2$ |

A uniform throw hits each color in proportion to its area. These are the theoretical probabilities, fixed before the first throw:

$$P(\text{Red}) = 0.50 \qquad P(\text{Blue}) = 0.30 \qquad P(\text{Green}) = 0.20$$

Now throw for real and keep score. After $n$ throws, the observed probability of a color is

$$\hat{P}(\text{color}) = \frac{\text{number of hits on that color}}{n}$$

Theoretical probability states what to expect. Observed probability records what happened. With few throws, $\hat{P}$ swings widely and can sit far from $P$. As $n$ grows, the Law of Large Numbers guarantees convergence toward the area ratios:

$$\hat{P}(\text{color}) \longrightarrow P(\text{color}) \quad \text{as } n \to \infty$$

<a id="2"></a>
## 2. Diagram: The Wall, Live

Each button throws at the wall and records where the stone lands. The table compares observed shares against the theoretical ones, and the right-hand chart tracks how the running frequencies settle onto their dashed targets as throws accumulate.

In [2]:
# The wall is drawn as a 1x1 square sliced into vertical strips whose widths
# equal the theoretical probabilities, so area fraction == probability.
colors = ['Red', 'Blue', 'Green']
theoretical_probs = {'Red': 0.50, 'Blue': 0.30, 'Green': 0.20}
color_hex = {'Red': '#e63946', 'Blue': '#3d8bfd', 'Green': '#2fb56a'}

assert abs(sum(theoretical_probs.values()) - 1.0) < 1e-9, "Probabilities must sum to 1"

boundaries = [0.0]
cum = 0.0
for c in colors:
    cum += theoretical_probs[c]
    boundaries.append(cum)
boundaries[-1] = 1.0  # guard against float drift


def get_color_for_point(x, y):
    """Return which colored region contains point (x, y)."""
    for i, c in enumerate(colors):
        lo, hi = boundaries[i], boundaries[i + 1]
        if (lo <= x < hi) or (i == len(colors) - 1 and x <= hi):
            return c
    return colors[-1]


state = {
    'total': 0,
    'hits': {c: 0 for c in colors},
    'points': [],   # landings recorded in throw order as (x, y, color)
}

rng = np.random.default_rng(42)


def throw_once():
    x, y = rng.uniform(0, 1), rng.uniform(0, 1)
    color = get_color_for_point(x, y)
    state['points'].append((x, y, color))
    state['hits'][color] += 1
    state['total'] += 1


throw_button = widgets.Button(description='Throw Ball', icon='bullseye',
                              button_style='success', layout=widgets.Layout(width='140px'))
throw10_button = widgets.Button(description='Throw x10', button_style='info',
                                layout=widgets.Layout(width='120px'))
throw100_button = widgets.Button(description='Throw x100', button_style='info',
                                 layout=widgets.Layout(width='120px'))
reset_button = widgets.Button(description='Reset', icon='refresh', button_style='danger',
                              layout=widgets.Layout(width='120px'))

wall_output = widgets.Output()
chart_output = widgets.Output()
table_output = widgets.HTML()


def render_wall():
    with wall_output:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(5.2, 5.2))

        for i, c in enumerate(colors):
            width = boundaries[i + 1] - boundaries[i]
            rect = patches.Rectangle((boundaries[i], 0), width, 1,
                                     facecolor=color_hex[c], edgecolor='white', linewidth=2)
            ax.add_patch(rect)
            ax.text(boundaries[i] + width / 2, 0.92, f"{c}",
                    ha='center', va='center', fontsize=12, fontweight='bold', color='white')
            ax.text(boundaries[i] + width / 2, 0.06, f"{theoretical_probs[c]*100:.0f}%",
                    ha='center', va='center', fontsize=11, fontweight='bold', color='white')

        if state['points']:
            xs = [p[0] for p in state['points']]
            ys = [p[1] for p in state['points']]
            ax.scatter(xs, ys, c='black', s=18, alpha=0.55, edgecolors='white',
                       linewidths=0.4, zorder=5)
            lx, ly, _ = state['points'][-1]
            ax.scatter([lx], [ly], c='gold', s=220, marker='*', edgecolors='black',
                       linewidths=1.3, zorder=6)

        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f"The Wall | Total throws: {state['total']}",
                     fontsize=12, fontweight='bold')
        for spine in ax.spines.values():
            spine.set_linewidth(2)

        plt.tight_layout()
        plt.show()


def render_charts():
    with chart_output:
        clear_output(wait=True)
        fig, (ax_bar, ax_conv) = plt.subplots(1, 2, figsize=(11, 4.3))

        # bar chart: theoretical vs observed
        xpos = np.arange(len(colors))
        w = 0.35
        theo_vals = [theoretical_probs[c] * 100 for c in colors]
        obs_vals = [(state['hits'][c] / state['total'] * 100) if state['total'] > 0 else 0
                    for c in colors]

        ax_bar.bar(xpos - w / 2, theo_vals, w, label='Theoretical',
                   color=[color_hex[c] for c in colors], alpha=0.35, edgecolor='black')
        ax_bar.bar(xpos + w / 2, obs_vals, w, label='Observed',
                   color=[color_hex[c] for c in colors], alpha=0.95, edgecolor='black')
        ax_bar.set_xticks(xpos)
        ax_bar.set_xticklabels(colors)
        ax_bar.set_ylabel('Probability (%)')
        ax_bar.set_ylim(0, max(65, max(theo_vals + obs_vals) + 10))
        ax_bar.set_title('Theoretical vs. Observed')
        ax_bar.legend(fontsize=8)
        for b in ax_bar.patches:
            h = b.get_height()
            ax_bar.annotate(f'{h:.1f}%', (b.get_x() + b.get_width() / 2, h),
                            textcoords='offset points', xytext=(0, 3), ha='center', fontsize=7)

        # convergence line chart (Law of Large Numbers)
        n = state['total']
        if n > 0:
            hist_colors = [p[2] for p in state['points']]
            t = np.arange(1, n + 1)
            for c in colors:
                indicator = np.array([1 if col == c else 0 for col in hist_colors])
                running_prob = np.cumsum(indicator) / t
                ax_conv.plot(t, running_prob * 100, color=color_hex[c],
                             linewidth=1.6, label=c)
                ax_conv.axhline(theoretical_probs[c] * 100, color=color_hex[c],
                                linestyle='--', alpha=0.5, linewidth=1)
            ax_conv.set_xlabel('Throw number')
            ax_conv.set_ylabel('Observed probability (%)')
            ax_conv.set_title('Convergence (Law of Large Numbers)')
            ax_conv.legend(fontsize=8, loc='upper right')
            ax_conv.set_ylim(0, 100)
        else:
            ax_conv.text(0.5, 0.5, 'Throw some balls to see\nconvergence over time',
                         ha='center', va='center', transform=ax_conv.transAxes,
                         fontsize=11, color='gray')
            ax_conv.set_xticks([])
            ax_conv.set_yticks([])
            ax_conv.set_title('Convergence (Law of Large Numbers)')

        plt.tight_layout()
        plt.show()


def render_table():
    rows = ''
    for c in colors:
        theo = theoretical_probs[c] * 100
        obs = (state['hits'][c] / state['total'] * 100) if state['total'] > 0 else 0
        diff = obs - theo
        diff_color = '#2fb56a' if abs(diff) < 3 else '#e6a23c' if abs(diff) < 8 else '#e63946'
        rows += f"""
        <tr>
            <td style="padding:8px 12px; border-bottom:1px solid #e5e5e5;">
                <span style="display:inline-block;width:11px;height:11px;border-radius:50%;
                             background:{color_hex[c]};margin-right:7px;"></span>{c}
            </td>
            <td style="padding:8px 12px; text-align:right; border-bottom:1px solid #e5e5e5;">{theo:.1f}%</td>
            <td style="padding:8px 12px; text-align:right; border-bottom:1px solid #e5e5e5;">{obs:.1f}%</td>
            <td style="padding:8px 12px; text-align:right; border-bottom:1px solid #e5e5e5;">{state['hits'][c]}</td>
            <td style="padding:8px 12px; text-align:right; border-bottom:1px solid #e5e5e5; color:{diff_color}; font-weight:600;">{diff:+.1f}%</td>
        </tr>"""

    table_output.value = f"""
    <div style="font-family: -apple-system, Segoe UI, Roboto, sans-serif;">
    <table style="border-collapse:collapse; width:100%; font-size:14px; margin-top:4px;">
        <thead>
            <tr style="background:#2c3e50; color:white;">
                <th style="padding:8px 12px; text-align:left;">Color</th>
                <th style="padding:8px 12px; text-align:right;">Theoretical</th>
                <th style="padding:8px 12px; text-align:right;">Observed</th>
                <th style="padding:8px 12px; text-align:right;">Hits</th>
                <th style="padding:8px 12px; text-align:right;">Diff</th>
            </tr>
        </thead>
        <tbody>{rows}</tbody>
    </table>
    <p style="font-size:13px; color:#666; margin-top:10px;">
        Total throws: <b>{state['total']}</b>
    </p>
    </div>
    """


def render_all():
    render_wall()
    render_charts()
    render_table()


def on_throw(_):
    throw_once()
    render_all()


def on_throw10(_):
    for _ in range(10):
        throw_once()
    render_all()


def on_throw100(_):
    for _ in range(100):
        throw_once()
    render_all()


def on_reset(_):
    state['total'] = 0
    state['hits'] = {c: 0 for c in colors}
    state['points'] = []
    render_all()


throw_button.on_click(on_throw)
throw10_button.on_click(on_throw10)
throw100_button.on_click(on_throw100)
reset_button.on_click(on_reset)

button_row = widgets.HBox([throw_button, throw10_button, throw100_button, reset_button])
top_row = widgets.HBox([wall_output, table_output],
                       layout=widgets.Layout(align_items='flex-start',
                                             justify_content='space-between'))

ui = widgets.VBox([button_row, top_row, chart_output])

render_all()
display(ui)

Click *Throw Ball* a few times and read the Diff column. The observed percentages can sit far from 50 / 30 / 20, especially early on. With few trials, randomness dominates and large gaps are normal.

Click *Throw x100* repeatedly. The lines in the convergence chart settle onto their dashed theoretical lines, and the Diff column shrinks toward 0%. Reset clears the experiment back to zero throws.

This behavior is the Law of Large Numbers: sample frequencies approach the true probabilities only in the long run. Any single short run can look very different from theory, which is why small samples deserve caution.

<a id="3"></a>
## 3. Example: Counting Outcomes Instead of Area

On the wall, probability was measured with geometry. When outcomes form a finite list, area becomes a count, and the same ratio gives the naive definition.

Roll a fair six-sided die and ask for an even number. Three of the six faces qualify:

$$P(\text{even}) = \frac{|\{2, 4, 6\}|}{|\{1, 2, 3, 4, 5, 6\}|} = \frac{3}{6} = \frac{1}{2}$$

Draw one card from a standard 52-card deck and ask for a heart. Thirteen of the 52 cards qualify:

$$P(\text{heart}) = \frac{13}{52} = \frac{1}{4}$$

In [3]:
faces = {1, 2, 3, 4, 5, 6}
evens = {2, 4, 6}
p_even = len(evens) / len(faces)

deck = [(rank, suit) for rank in range(1, 14) for suit in range(4)]
hearts = [card for card in deck if card[1] == 0]
p_heart = len(hearts) / len(deck)

print("die : |A| =", len(evens), "  |S| =", len(faces), "  P(even) =", p_even)
print("card: |A| =", len(hearts), " |S| =", len(deck), "  P(heart) =", p_heart)

die : |A| = 3   |S| = 6   P(even) = 0.5
card: |A| = 13  |S| = 52   P(heart) = 0.25


The naive values are exact fractions. A simulation checks them the way the wall widget did, by counting what actually happens over many trials:

In [4]:
rng = np.random.default_rng(7)
n = 100_000

rolls = rng.integers(1, 7, size=n)
suits = rng.integers(0, 4, size=n)

print(f"observed P(even)  = {(rolls % 2 == 0).mean():.4f}   naive = {p_even}")
print(f"observed P(heart) = {(suits == 0).mean():.4f}   naive = {p_heart:.4f}")

observed P(even)  = 0.4999   naive = 0.5
observed P(heart) = 0.2506   naive = 0.2500


<a id="4"></a>
## 4. Plain English

When every outcome is equally likely, the probability of an event is the number of ways it can happen divided by the total number of possible outcomes. On the wall those outcomes were points, measured by area. With dice and cards they are separate cases, measured by counting.

<a id="5"></a>
## 5. Technical Definitions

#### Definition 1.3.1 (Naive definition of probability)

Let $A$ be an event for an experiment with a finite sample space $S$. The naive probability of $A$ is

$$P_{\text{naive}}(A) = \frac{|A|}{|S|} = \frac{\text{number of outcomes favorable to } A}{\text{total number of outcomes in } S}$$

Here $|A|$ denotes the size of $A$. The definition assumes that $S$ is finite and non-empty and that all outcomes are equally likely. *(Blitzstein)*

#### Key Limitations

The naive definition requires a finite sample space and assigns the same probability mass to every outcome in it. Both requirements are restrictive: many experiments have infinitely many outcomes, and in many others the outcomes are not equally likely.

#### Conditions for Applicability

* **Physical or mathematical symmetry:** symmetries make all outcomes equally likely. A fair coin lands Heads half the time because of physical symmetry, and a well-shuffled standard deck makes every ordering equally probable (Blitzstein & Hwang, 2019).
* **Equally likely outcomes by design:** randomized mechanisms can force equal likelihood. When drawing a simple random sample of size $n$ from a population of size $N$, the sampling design makes all $\binom{N}{n}$ subsets of size $n$ equally likely (Blitzstein & Hwang, 2019).
* **As a baseline or null model:** equal likelihood can be assumed on purpose so that a theory produces predicted values to compare against data (Blitzstein & Hwang, 2019).

<a id="6"></a>
## 6. References

- Blitzstein, J. K., & Hwang, J. (2019). *Introduction to Probability* (2nd ed.). CRC Press.